# 01 — MDP Fundamentals

Reinforcement learning is the study of how an agent should act to maximise cumulative reward. The Markov Decision Process (MDP) is the mathematical framework that formalises this. Everything in RL — from classic algorithms to modern deep RL to RLHF — is a solution method for some variant of an MDP.

## The five elements of an MDP

| Symbol | Name | Meaning |
|--------|------|---------|
| **S** | State space | All possible situations the agent can be in |
| **A** | Action space | All possible actions the agent can take |
| **P(s'\|s,a)** | Transition function | Probability of reaching s' after taking a in s |
| **R(s,a,s')** | Reward function | Scalar signal received after each transition |
| **γ** | Discount factor | How much future rewards are worth relative to immediate ones |

The **Markov property**: the future is conditionally independent of the past given the present state. The transition probability P(s'|s,a) depends only on s and a, not on how the agent got to s.

This is an assumption. Real-world problems often violate it (partial observability, non-stationarity), and much of advanced RL is about relaxing it.

## What the agent optimises

The agent's goal is to find a **policy** π(a|s) — a mapping from states to actions (or a distribution over actions) — that maximises the **expected discounted return**:

$$G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

The discount factor γ ∈ [0,1) serves two purposes:
1. Mathematical convenience (makes infinite-horizon sums finite)
2. Encodes preference for sooner rewards — important in practice because future rewards are uncertain

With γ=0 the agent is greedy (only cares about immediate reward). With γ→1 the agent is far-sighted (treats all future rewards nearly equally).

## Value functions

Two key functions characterise how good a state or action is under a policy:

**State-value function** V^π(s): expected return starting from state s, following policy π
$$V^\pi(s) = \mathbb{E}_\pi\left[G_t \mid S_t = s\right]$$

**Action-value function** Q^π(s,a): expected return starting from state s, taking action a, then following π
$$Q^\pi(s,a) = \mathbb{E}_\pi\left[G_t \mid S_t = s, A_t = a\right]$$

The **Bellman equation** expresses V^π recursively — it is the foundation of nearly every RL algorithm:
$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma V^\pi(s')\right]$$

In [ ]:
# GridWorld MDP from scratch
# A 4x4 grid, terminal states at (0,0) and (3,3)
# Actions: up, down, left, right — deterministic transitions
# Reward: -1 per step, 0 at terminal states

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

class GridWorld:
    def __init__(self, size=4, gamma=0.9):
        self.size = size
        self.gamma = gamma
        self.n_states = size * size
        self.n_actions = 4  # 0=up, 1=down, 2=left, 3=right
        self.terminals = {0, self.n_states - 1}  # top-left and bottom-right
        self._build_transitions()

    def _build_transitions(self):
        n, m = self.size, self.n_actions
        # P[s, a] = list of (prob, next_state, reward, done)
        self.P = {}
        for s in range(n * n):
            self.P[s] = {}
            row, col = s // n, s % n
            for a in range(m):
                if s in self.terminals:
                    self.P[s][a] = [(1.0, s, 0.0, True)]
                    continue
                dr, dc = [(-1,0),(1,0),(0,-1),(0,1)][a]
                nr = max(0, min(n-1, row + dr))
                nc = max(0, min(n-1, col + dc))
                ns = nr * n + nc
                self.P[s][a] = [(1.0, ns, -1.0, ns in self.terminals)]

    def policy_evaluation(self, policy, theta=1e-6, max_iter=1000):
        """Iterative policy evaluation — compute V^π for a given policy.""""
        V = np.zeros(self.n_states)
        for i in range(max_iter):
            delta = 0
            for s in range(self.n_states):
                v = V[s]
                new_v = 0
                for a, prob in enumerate(policy[s]):
                    for p, ns, r, _ in self.P[s][a]:
                        new_v += prob * p * (r + self.gamma * V[ns])
                V[s] = new_v
                delta = max(delta, abs(v - V[s]))
            if delta < theta:
                print(f"  Policy evaluation converged in {i+1} iterations (Δ={delta:.2e})")
                break
        return V

    def policy_improvement(self, V):
        """Greedy policy improvement given V.""""
        policy = np.zeros((self.n_states, self.n_actions))
        for s in range(self.n_states):
            if s in self.terminals:
                policy[s] = 1.0 / self.n_actions
                continue
            q_values = []
            for a in range(self.n_actions):
                q = sum(p * (r + self.gamma * V[ns]) for p, ns, r, _ in self.P[s][a])
                q_values.append(q)
            best = np.argmax(q_values)
            policy[s, best] = 1.0
        return policy

    def policy_iteration(self):
        """Full policy iteration: alternates evaluation and improvement until stable.""""
        # Start with random policy
        policy = np.ones((self.n_states, self.n_actions)) / self.n_actions
        for iteration in range(100):
            V = self.policy_evaluation(policy)
            new_policy = self.policy_improvement(V)
            if np.allclose(policy, new_policy):
                print(f"Policy iteration converged in {iteration+1} cycles")
                return V, new_policy
            policy = new_policy
        return V, policy

env = GridWorld(size=4, gamma=0.9)
V_star, pi_star = env.policy_iteration()

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Value function heatmap
ax = axes[0]
V_grid = V_star.reshape(4, 4)
im = ax.imshow(V_grid, cmap='RdYlGn', vmin=V_star.min(), vmax=0)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{V_grid[i,j]:.1f}', ha='center', va='center', fontsize=11, fontweight='bold')
ax.set_title('Optimal Value Function V*(s)', fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(im, ax=ax)

# Optimal policy arrows
ax2 = axes[1]
ax2.set_xlim(-0.5, 3.5); ax2.set_ylim(-0.5, 3.5)
ax2.set_aspect('equal')
action_arrows = ['^', 'v', '<', '>']
for s in range(16):
    row, col = s // 4, s % 4
    if s in env.terminals:
        ax2.add_patch(patches.Rectangle((col-0.4, 3-row-0.4), 0.8, 0.8, color='gold'))
        ax2.text(col, 3-row, 'T', ha='center', va='center', fontsize=14, fontweight='bold')
    else:
        best_a = np.argmax(pi_star[s])
        ax2.text(col, 3-row, action_arrows[best_a], ha='center', va='center', fontsize=16)
ax2.set_title('Optimal Policy π*(s)', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(4)); ax2.set_yticks(range(4))

plt.tight_layout()
plt.savefig('/tmp/gridworld.png', dpi=100, bbox_inches='tight')
plt.show()
print("GridWorld: optimal policy and value function computed via policy iteration")


In [ ]:
# Explore the effect of discount factor γ on the optimal value function

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, gamma in zip(axes, [0.5, 0.9, 0.99]):
    env_g = GridWorld(size=4, gamma=gamma)
    V, pi = env_g.policy_iteration()
    V_grid = V.reshape(4, 4)
    vmin = V.min()
    im = ax.imshow(V_grid, cmap='RdYlGn', vmin=vmin, vmax=0)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{V_grid[i,j]:.1f}', ha='center', va='center', fontsize=10)
    ax.set_title(f'γ = {gamma}', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax)

plt.suptitle('Effect of Discount Factor on V*(s)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/gamma_effect.png', dpi=100, bbox_inches='tight')
plt.show()
print("Higher γ → more negative values (agent plans further ahead, accumulates more -1 steps in its estimate)")


## Key takeaways

- The MDP formalism is universal — RLHF, game-playing, robotics, and recommendation systems are all MDPs with different S, A, P, R, γ
- Policy iteration (evaluation + improvement) is provably convergent to the optimal policy — but requires a known transition model P, which is usually not available in practice
- The need to learn P (or avoid needing it) is what motivates model-free RL (next notebooks)